In [1]:
import os
import sys
import torch
import numpy as np
import pyro
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import Adam

# Ajuste de ruta para ver 'src'
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

root_path = os.getcwd()
if root_path not in sys.path:
    sys.path.append(root_path)

from src.data.loader import make_dataloaders
from src.models.lda import TopicModeler
from src.models.bnn import BayesianClassifier
from src.inference.variational import VariationalInference

# Limpiar el estado de Pyro por si acaso
pyro.clear_param_store()

In [2]:
import random
import numpy as np
import torch
import pyro

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
pyro.set_rng_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
# # 1. Cargar el modelo LDA que acabas de entrenar
# lda_modeler = TopicModeler.load("experiments/results/lda_model.pkl")

# # 2. Cargar TF-IDF y generar los vectores theta (Dirichlet) para todo el dataset
# from src.data.loader import load_tfidf_splits
# X_tr, X_val, X_te, _ = load_tfidf_splits()
# theta_tr = lda_modeler.get_topics(X_tr)
# theta_val = lda_modeler.get_topics(X_val)
# theta_te = lda_modeler.get_topics(X_te)

# # Concatenamos todos para que el loader sepa repartirlos
# all_theta = np.vstack([theta_tr, theta_val, theta_te])

# 3. Crear DataLoaders (Aquí se combinan BERT + LDA)
train_loader, val_loader, test_loader = make_dataloaders(
    batch_size=64, 
    # topic_vecs=all_theta
)

# Verificar que los DataLoaders funcionan correctamente
x_batch, y_batch = next(iter(train_loader))
print("Train batch X shape:", x_batch.shape)
print("Train batch y shape:", y_batch.shape)

assert x_batch.shape[1] == 768, f"Expected input_dim=778, got {x_batch.shape[1]}"
print("Input dimension check passed.")

Train batch X shape: torch.Size([64, 768])
Train batch y shape: torch.Size([64])
Input dimension check passed.


In [4]:
# Dimensiones: 768 (BERT) + 10 (LDA)
lda_dim = 10
bert_dim = 768
expected_input_dim = bert_dim + lda_dim
input_dim = bert_dim
model = BayesianClassifier(input_dim=input_dim)
vi_engine = VariationalInference(model, lr=3e-4)

print(f"Modelo inicializado con input_dim={input_dim}")

Modelo inicializado con input_dim=768


In [5]:
import torch
from pyro.infer import Predictive

def evaluate_bnn_metrics(model, guide, data_loader, num_mc_samples=30, device="cpu"):
    model.eval()

    all_probs = []
    all_targets = []

    predictive = Predictive(
        model,
        guide=guide,
        num_samples=num_mc_samples,
        return_sites=["_RETURN"]
    )

    with torch.no_grad():
        for x_batch, y_batch in data_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            sampled_logits = predictive(x_batch)["_RETURN"]   # shape: [S, B, 2]
            sampled_probs = torch.softmax(sampled_logits, dim=-1)
            mean_probs = sampled_probs.mean(dim=0)            # shape: [B, 2]

            all_probs.append(mean_probs.cpu())
            all_targets.append(y_batch.cpu())

    all_probs = torch.cat(all_probs, dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    preds = all_probs.argmax(dim=1)
    accuracy = (preds == all_targets).float().mean().item()

    eps = 1e-8
    true_class_probs = all_probs[torch.arange(len(all_targets)), all_targets]
    nll = -torch.log(true_class_probs.clamp(min=eps)).mean().item()

    return {
        "accuracy": accuracy,
        "nll": nll,
    }

In [6]:
best_val_nll = float("inf")
best_val_elbo = float("inf")
best_epoch = -1
best_param_path = "experiments/results/bnn_params_best.pt"

patience = 5
patience_counter = 0
min_delta = 1e-4

epochs = 30
train_losses = []
val_losses = []
val_accuracies = []
val_nlls = []

os.makedirs("experiments/results", exist_ok=True)

for epoch in range(epochs):
    total_loss = 0.0
    total_samples = 0

    for x_batch, y_batch in train_loader:
        batch_loss = vi_engine.train_step(x_batch, y_batch)   # average loss per sample
        total_loss += batch_loss * x_batch.shape[0]
        total_samples += x_batch.shape[0]

    avg_loss = total_loss / total_samples
    train_losses.append(avg_loss)

    # Validation ELBO
    val_loss = vi_engine.evaluate_loss(val_loader)
    val_losses.append(val_loss)

    # Validation predictive metrics
    val_metrics = evaluate_bnn_metrics(
        model=vi_engine.model,
        guide=vi_engine.guide,
        data_loader=val_loader,
        num_mc_samples=30
    )

    val_acc = val_metrics["accuracy"]
    val_nll = val_metrics["nll"]

    val_accuracies.append(val_acc)
    val_nlls.append(val_nll)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train ELBO: {avg_loss:.4f} | "
        f"Val ELBO: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val NLL: {val_nll:.4f}"
    )

    # Save best model according to validation NLL
    if val_nll < best_val_nll - min_delta:
        best_val_nll = val_nll
        best_val_elbo = val_loss
        best_epoch = epoch + 1
        patience_counter = 0
        pyro.get_param_store().save(best_param_path)
        print(f"New best model saved at epoch {best_epoch}")
    else:
        patience_counter += 1
        print(f"No improvement. Early stopping counter: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        break

# Save last model separately
pyro.get_param_store().save("experiments/results/bnn_params_last.pt")

print("\nTraining finished.")
print(f"Best epoch: {best_epoch}")
print(f"Best validation NLL: {best_val_nll:.4f}")
print(f"Best validation ELBO: {best_val_elbo:.4f}")
print(f"Best model saved to: {best_param_path}")
print("Last model saved to: experiments/results/bnn_params_last.pt")

Epoch 1/30 | Train ELBO: 2838.6089 | Val ELBO: 2871.0226 | Val Acc: 0.6344 | Val NLL: 0.6741
New best model saved at epoch 1
Epoch 2/30 | Train ELBO: 2776.7474 | Val ELBO: 2811.0471 | Val Acc: 0.6863 | Val NLL: 0.6101
New best model saved at epoch 2
Epoch 3/30 | Train ELBO: 2716.9220 | Val ELBO: 2747.5095 | Val Acc: 0.7153 | Val NLL: 0.5449
New best model saved at epoch 3
Epoch 4/30 | Train ELBO: 2658.1274 | Val ELBO: 2688.9420 | Val Acc: 0.7313 | Val NLL: 0.5144
New best model saved at epoch 4
Epoch 5/30 | Train ELBO: 2600.6223 | Val ELBO: 2629.9044 | Val Acc: 0.7662 | Val NLL: 0.4873
New best model saved at epoch 5
Epoch 6/30 | Train ELBO: 2543.2525 | Val ELBO: 2571.9167 | Val Acc: 0.7582 | Val NLL: 0.4839
New best model saved at epoch 6
Epoch 7/30 | Train ELBO: 2486.8980 | Val ELBO: 2515.9709 | Val Acc: 0.7562 | Val NLL: 0.4857
No improvement. Early stopping counter: 1/5
Epoch 8/30 | Train ELBO: 2431.2181 | Val ELBO: 2457.6623 | Val Acc: 0.7313 | Val NLL: 0.5010
No improvement. Earl